In [36]:
#imports

import pandas as pd
import numpy as np
import requests

In [24]:
#import data

data = pd.read_csv('Dataset4820.csv')
data

,Used Date,From,To,Airline,Scheduled Departure,SDEP,Departure,DEP,Scheduled Arrival,SARR,...,Market Share,OTP Index,weather__hourly__windspeedKmph,weather__hourly__weatherDesc__value,weather__hourly__precipMM,weather__hourly__humidity,weather__hourly__visibility,weather__hourly__pressure,weather__hourly__cloudcover,Category
0,31-01-2020,DEL,HYD,Air Asia,06:05,605,06:04,604,08:15,815,...,5.3,85.6,11,Partly cloudy,0,90,9,1012,57,0
1,31-01-2020,DEL,HYD,Indigo,04:50,450,04:36,436,07:00,700,...,47.5,81.1,11,Partly cloudy,0,90,9,1012,57,0
2,30-01-2020,DEL,HYD,Air Asia,06:05,605,05:54,554,08:15,815,...,5.3,85.6,11,Partly cloudy,0,90,9,1012,57,0
3,30-01-2020,DEL,HYD,Air Asia,17:40,1740,17:40,1740,19:55,1955,...,5.3,85.6,11,Partly cloudy,0,90,9,1012,57,0
4,30-01-2020,DEL,HYD,Air Asia,19:50,1950,19:54,1954,22:10,2210,...,5.3,85.6,11,Partly cloudy,0,90,9,1012,57,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10713,24-01-2019,CCU,DEL,SpiceJet,18:30,1830,18:30,1830,20:55,2055,...,13.3,69.2,6,Clear,0,29,10,1016,1,0
10714,23-01-2019,CCU,DEL,SpiceJet,18:30,1830,18:30,1830,20:55,2055,...,13.3,69.2,3,Clear,0,27,10,1015,0,0
10715,22-01-2019,CCU,DEL,SpiceJet,18:30,1830,18:38,1838,20:55,2055,...,13.3,69.2,5,Clear,0,28,10,1015,0,1
10716,09-01-2019,DEL,HYD,Spicejet,22:05,2205,00:40,40,00:25,25,...,13.3,69.2,8,Patchy rain possible,1,55,9,997,85,4


In [25]:
print(len(data["From"].unique()))
#only four different airports it comes from

4


In [26]:
#Adds airport congestion with COUNT GROUPBY using airport 

data["Airport_Congestion"] = (
    data.groupby(["Used Date", "From", "Scheduled Departure"])["Airline"]
    .transform("count")
)

In [27]:
data["Departure_Hour"] = pd.to_datetime(
    data["Scheduled Departure"], format="%H:%M"
).dt.hour #gets hour

In [29]:
# Count flights per airport, date, and hour
hourly_counts = (
    data.groupby(["From", "Used Date", "Departure_Hour"])
    .size()
    .reset_index(name="Flight_Count")
)

# Calculate average flights per airport and hour
avg_congestion = (
    hourly_counts.groupby(["From", "Departure_Hour"])["Flight_Count"]
    .mean()
    .reset_index(name="Avg_Airport_Congestion")
)

# Merge back into original dataset
data = data.merge(
    avg_congestion,
    on=["From", "Departure_Hour"],
    how="left"
)

In [32]:
data["Status"].value_counts()
data = data[data["Status"] == 1].copy()
data.drop(columns=["Status"], inplace=True)

In [33]:
##TEST## Goal is to add features of weather hours before departure 

airports = {
    "DEL": (28.5562, 77.1000),
    "BOM": (19.0896, 72.8656),
    "BLR": (13.1986, 77.7066),
    "CCU": (22.6547, 88.4467)
}

weather_variables = [
    "temperature_2m",
    "wind_speed_10m",
    "precipitation",
    "relative_humidity_2m",
    "visibility",
    "pressure_msl",
    "cloud_cover",
    "weather_code"
]

In [ ]:
#uses ERA5 data
data["Scheduled_DateTime"] = pd.to_datetime(
    data["Used Date"].astype(str) + " " +
    data["Scheduled Departure"].astype(str),
    dayfirst=True,
    errors="coerce"
)

data["Weather_Cutoff"] = (
    data["Scheduled_DateTime"] - pd.Timedelta(hours=2)
)

start_date = data["Weather_Cutoff"].min().strftime("%Y-%m-%d")
end_date = data["Weather_Cutoff"].max().strftime("%Y-%m-%d")

weather_frames = []

for airport, (lat, lon) in airports.items():

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(weather_variables),
        "timezone": "Asia/Kolkata",
        "models": "era5"
    }

    response = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params=params,
        timeout=120
    )
    response.raise_for_status()

    weather = pd.DataFrame(response.json()["hourly"])
    weather["From"] = airport

    weather_frames.append(weather)

weather_data = pd.concat(weather_frames, ignore_index=True)
weather_data["time"] = pd.to_datetime(weather_data["time"])

print("Downloaded weather rows:", len(weather_data))

Downloaded weather rows: 38016


In [38]:
data = pd.merge_asof(
    data.sort_values("Weather_Cutoff"),
    weather_data.sort_values("time"),
    left_on="Weather_Cutoff",
    right_on="time",
    by="From",
    direction="backward",
    tolerance=pd.Timedelta(hours=1)
)

In [39]:
# Rename new weather columns without touching original columns
weather_mapping = {
    "wind_speed_10m": "Past_windspeedKmph",
    "precipitation": "Past_precipMM",
    "relative_humidity_2m": "Past_humidity",
    "visibility": "Past_visibility",
    "pressure_msl": "Past_pressure",
    "cloud_cover": "Past_cloudcover",
    "weather_code": "Past_weatherDesc"
}

# Convert visibility from meters to kilometers
data["visibility"] = data["visibility"] / 1000

# Rename new historical weather features
data = data.rename(columns=weather_mapping)

# Optional: keep historical temperature too
data = data.rename(columns={
    "temperature_2m": "Past_temperature"
})

In [40]:
original_cols = [
    col for col in data.columns
    if col.startswith("weather__hourly__")
]

past_cols = [
    col for col in data.columns
    if col.startswith("Past_")
]

print(data[original_cols + past_cols].head())

print("\nMissing historical weather:")
print(data[past_cols].isna().sum())

print("\nTotal flights:", len(data))

   weather__hourly__windspeedKmph weather__hourly__weatherDesc__value  \
0                               8                               Clear   
1                               8                Patchy rain possible   
2                               5                               Clear   
3                               3                               Clear   
4                               6                               Clear   

   weather__hourly__precipMM  weather__hourly__humidity  \
0                          0                         47   
1                          1                         55   
2                          0                         28   
3                          0                         27   
4                          0                         29   

   weather__hourly__visibility  weather__hourly__pressure  \
0                           10                       1018   
1                            9                        997   
2                      

In [41]:
data.to_csv("Dataset4820_Both_Weather.csv", index=False)

In [42]:
data = data.drop(columns=["Past_visibility"])

In [43]:
print(
    data["Avg_Airport_Congestion_x"].equals(
        data["Avg_Airport_Congestion_y"]
    )
)

True


In [44]:
data = data.drop_duplicates().copy()